# RL Foundations: MDPs, Value Functions, and Dynamic Programming


Reinforcement learning provides the mathematical framework underlying every alignment algorithm used to train modern language models. Where supervised learning tells a model the correct output, RL teaches it to maximize cumulative reward through trial and error — a fundamentally different mode of learning that enables behaviors no labeled dataset could specify.

This notebook builds the framework from the ground up. We define the **Markov decision process** (MDP) as the formal model of sequential decision-making, derive the **Bellman equations** that characterize optimal behavior, implement **dynamic programming** algorithms that solve small MDPs exactly, and introduce **model-free** methods (Monte Carlo, TD learning, Q-learning) that learn from experience alone.

One thread running through every section is the **advantage function** $A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s).$ This quantity — how much better action $a$ is than average under policy $\pi$ — is the signal GRPO uses to update language models. Understanding where it comes from is the purpose of this notebook.

**Prerequisites.** Basic Python and NumPy, some calculus. No prior RL experience assumed. The GRPO tutorial is the forward reference for how these ideas map onto language model training.


## The Reinforcement Learning Problem


At each discrete time step $t$, an **agent** observes the current **state** $S_t \in \mathcal{S}$, selects an **action** $A_t \in \mathcal{A}$, and receives a scalar **reward** $R_{t+1} \in \mathbb{R}$ from the **environment**, which also transitions to a new state $S_{t+1}.$ This loop repeats, generating a **trajectory**

$$\tau = (S_0, A_0, R_1, S_1, A_1, R_2, S_2, \ldots).$$

The agent's goal is to accumulate large total reward over time. Rather than treating all future rewards equally, we discount them geometrically with factor $\gamma \in [0, 1)$, which captures the preference for sooner rewards and ensures convergence for infinite-horizon problems. The **return** from time $t$ is

$$G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}.$$

The agent's behavior is specified by a **policy** $\pi(a \mid s)$, a conditional probability distribution over actions given states. When the policy is deterministic we write $\pi(s) = a.$ The **RL objective** is to find a policy that maximizes expected return from the initial state:

$$\max_\pi \; \mathbb{E}_\pi[G_0].$$

**Example.** A language model is a policy: the **state** is the prompt (and any tokens generated so far), the **action** is the next token, and the **reward** comes from a reward model that evaluates the quality of the completed response. The policy improvement algorithms that shape modern LLMs — RLHF, PPO, GRPO — are all instances of the optimization above applied to this specific MDP structure. We return to this connection in a [RL4: RL for Language Models](./04-rl-for-llms.html).


## Markov Decision Processes


A **Markov decision process** (MDP) is a tuple $\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, P, R, \gamma \rangle$ where $\mathcal{S}$ is the state space, $\mathcal{A}$ is the action space, $P(s' \mid s, a)$ is the **transition kernel**, $R(s, a)$ is the expected reward, and $\gamma$ is the discount factor. The **Markov property** states that the future depends on the past only through the current state:

$$P(S_{t+1} = s' \mid S_0, A_0, \ldots, S_t, A_t) = P(S_{t+1} = s' \mid S_t, A_t).$$

This memoryless structure is what makes the Bellman equations tractable — the value of a state is determined entirely by what can happen from that state onward, not by how we arrived there.

We work throughout with a concrete environment: a $4 \times 4$ **GridWorld**. States $0$–$15$ are numbered row-major. The agent takes actions in $\{\uparrow, \rightarrow, \downarrow, \leftarrow\}$ and receives a reward of $-1$ per step, $+10$ upon reaching the goal (state $15$, terminal), and $-10$ upon falling into the trap (state $5$, terminal). Moving into a wall keeps the agent in place.


**Setup.** We import all dependencies and configure SVG output:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

np.random.seed(42)


**GridWorld.** We implement the $4 \times 4$ GridWorld MDP from scratch. The `_build_dynamics` method constructs the full transition tensor $P[s, a, s']$ and reward matrix $R[s, a]$, which are used directly by the dynamic programming algorithms in later sections:


In [ ]:
class GridWorld:
    """4x4 GridWorld MDP. States 0-15 (row-major). Goal=15, Trap=5."""

    NROWS = 4
    NCOLS = 4
    NSTATES = 16
    NACTIONS = 4
    ACTION_SYMBOLS = ["↑", "→", "↓", "←"]

    # Action deltas: up, right, down, left (row, col)
    _DELTAS = [(-1, 0), (0, 1), (1, 0), (0, -1)]

    def __init__(self, gamma=0.99):
        self.gamma = gamma
        self.goal = 15
        self.trap = 5
        self.terminals = {self.goal, self.trap}
        self._build_dynamics()

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _rc(self, s):
        """State index -> (row, col)."""
        return divmod(s, self.NCOLS)

    def _s(self, r, c):
        """(row, col) -> state index."""
        return r * self.NCOLS + c

    # ------------------------------------------------------------------
    # Dynamics
    # ------------------------------------------------------------------

    def _build_dynamics(self):
        """Construct P[s, a, s'] and R[s, a] arrays."""
        S, A = self.NSTATES, self.NACTIONS
        P = np.zeros((S, A, S))
        R = np.zeros((S, A))

        for s in range(S):
            if s in self.terminals:
                # Terminal states are absorbing: all actions stay in s, reward 0
                for a in range(A):
                    P[s, a, s] = 1.0
                    R[s, a] = 0.0
                continue

            r, c = self._rc(s)
            for a, (dr, dc) in enumerate(self._DELTAS):
                nr, nc = r + dr, c + dc
                # Clamp to grid boundaries (wall = stay in place)
                nr = max(0, min(self.NROWS - 1, nr))
                nc = max(0, min(self.NCOLS - 1, nc))
                s_next = self._s(nr, nc)

                P[s, a, s_next] += 1.0

                # Reward depends on the state we land in
                if s_next == self.goal:
                    R[s, a] += 10.0
                elif s_next == self.trap:
                    R[s, a] += -10.0
                else:
                    R[s, a] += -1.0

        self.P = P  # (NSTATES, NACTIONS, NSTATES)
        self.R = R  # (NSTATES, NACTIONS)

    # ------------------------------------------------------------------
    # Simulator
    # ------------------------------------------------------------------

    def reset(self, start=0):
        """Reset to start state (default top-left corner)."""
        self.state = start
        return self.state

    def step(self, s, a):
        """Sample next state from P[s, a, :]. Return (s', r, done)."""
        probs = self.P[s, a]
        s_next = int(np.random.choice(self.NSTATES, p=probs))
        r = self.R[s, a]
        done = s_next in self.terminals
        return s_next, r, done

    # ------------------------------------------------------------------
    # Rendering
    # ------------------------------------------------------------------

    def render_values(self, V, title="Value Function", ax=None):
        """Heatmap of value function on the 4x4 grid."""
        show = ax is None
        if ax is None:
            _, ax = plt.subplots(figsize=(4, 4))

        grid = V.reshape(self.NROWS, self.NCOLS)
        im = ax.imshow(grid, cmap="RdYlGn", vmin=V.min(), vmax=V.max())

        for s in range(self.NSTATES):
            r, c = self._rc(s)
            label = f"{V[s]:.1f}"
            if s == self.goal:
                label = f"G\n{V[s]:.1f}"
            elif s == self.trap:
                label = f"T\n{V[s]:.1f}"
            ax.text(c, r, label, ha="center", va="center", fontsize=8,
                    color="black")

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title, fontsize=10)
        if show:
            plt.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()
        return im

    def render_policy(self, pi, V=None, title="Policy", ax=None):
        """Grid with arrows for policy; optionally overlay value heatmap."""
        show = ax is None
        if ax is None:
            _, ax = plt.subplots(figsize=(4, 4))

        if V is not None:
            grid = V.reshape(self.NROWS, self.NCOLS)
            ax.imshow(grid, cmap="RdYlGn", vmin=V.min(), vmax=V.max())
        else:
            ax.imshow(np.zeros((self.NROWS, self.NCOLS)), cmap="Blues",
                      vmin=0, vmax=1)

        # Arrow offsets for (up, right, down, left)
        arrow_dx = [0,  0.3, 0,   -0.3]
        arrow_dy = [-0.3, 0,  0.3,  0]

        for s in range(self.NSTATES):
            r, c = self._rc(s)
            if s in self.terminals:
                label = "G" if s == self.goal else "T"
                ax.text(c, r, label, ha="center", va="center",
                        fontsize=11, fontweight="bold", color="black")
                continue
            a = int(pi[s])
            ax.annotate("", xy=(c + arrow_dx[a], r + arrow_dy[a]),
                        xytext=(c, r),
                        arrowprops=dict(arrowstyle="->", color="black", lw=1.5))

        ax.set_xticks(range(self.NCOLS))
        ax.set_yticks(range(self.NROWS))
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_title(title, fontsize=10)
        ax.grid(linestyle="dotted", alpha=0.5)
        if show:
            plt.tight_layout()
            plt.show()


**Rollouts.** We run a few random rollouts to verify the environment dynamics:


In [ ]:
env = GridWorld(gamma=0.99)
np.random.seed(42)

for episode in range(3):
    s = env.reset(start=0)
    total_r = 0.0
    steps = 0
    while True:
        a = np.random.randint(env.NACTIONS)
        s_next, r, done = env.step(s, a)
        total_r += r
        steps += 1
        s = s_next
        if done or steps >= 50:
            break
    outcome = "goal" if s == env.goal else ("trap" if s == env.trap else "timeout")
    print(f"  Episode {episode+1}: {steps} steps, return={total_r:.1f}, outcome={outcome}")


**Dynamics shape check.** We confirm the transition and reward arrays have the expected shapes:


In [ ]:
print(f"P shape: {env.P.shape}   (states × actions × next_states)")
print(f"R shape: {env.R.shape}   (states × actions)")
print(f"Row sums of P (should be 1): {env.P.sum(axis=2).min():.4f} – {env.P.sum(axis=2).max():.4f}")


## Value Functions and Bellman Equations


A policy's quality is characterized by two functions. The **state-value function** measures expected return starting from state $s$ and acting under $\pi$ thereafter:

$$V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \;\ \Big|\;\ S_t = s\right].$$

The **action-value function** (or $Q$-function) fixes both the starting state and the first action:

$$Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a].$$

The two are related by averaging $Q$ over actions under $\pi$:

$$V^\pi(s) = \sum_a \pi(a \mid s)\, Q^\pi(s, a).$$


### Bellman Expectation Equations


Expanding $G_t = R_{t+1} + \gamma G_{t+1}$ and taking expectations yields the **Bellman expectation equations**:

$$\boxed{V^\pi(s) = \sum_a \pi(a \mid s)\left[R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, V^\pi(s')\right]}$$

$$Q^\pi(s, a) = R(s, a) + \gamma \sum_{s'} P(s'\mid s, a)\, V^\pi(s').$$

These are consistency conditions: a self-referential system of linear equations that $V^\pi$ must satisfy. For a finite MDP with $|\mathcal{S}| = N$ states, they form a linear system of $N$ equations in $N$ unknowns that can be solved exactly via matrix inversion.


### Advantage Function


The **advantage function** measures how much better action $a$ is relative to the average:

$$A^\pi(s, a) = Q^\pi(s, a) - V^\pi(s).$$

A positive advantage means action $a$ exceeds the policy's average; a negative advantage means it falls short. Note that $\sum_a \pi(a \mid s)\, A^\pi(s, a) = 0$ by construction — the average advantage is always zero.


:::{.callout-important}
The advantage $A^\pi(s,a)$ measures how much better action $a$ is than the average under $\pi$. In GRPO, the group-relative advantage $\hat{A}_i = (r_i - \bar{r})/\sigma_r$ is a Monte Carlo estimate of exactly this quantity — the "state" is the prompt, the "action" is the full response, and the reward comes from a reward model.

:::


### Bellman Optimality Equations


The **optimal value function** $V^*$ satisfies the **Bellman optimality equations**, obtained by replacing policy-weighted averaging with a max:

$$V^*(s) = \max_a Q^*(s, a), \qquad Q^*(s, a) = R(s, a) + \gamma \sum_{s'} P(s'\mid s, a)\, V^*(s').$$

Combined:

$$\boxed{V^*(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'\mid s, a)\, V^*(s')\right].}$$

The optimal policy is then recovered greedily: $\pi^*(s) = \operatorname{argmax}_a Q^*(s, a).$ Unlike the Bellman expectation equations, the optimality equations are nonlinear (due to the max), so we cannot solve them directly by matrix inversion.


### Exact Bellman Solve


For a fixed policy $\pi$, the Bellman expectation equation is linear in $V^\pi.$ Writing $P^\pi[s, s'] = \sum_a \pi(a\mid s) P(s'\mid s, a)$ and $R^\pi[s] = \sum_a \pi(a\mid s) R(s, a)$, we have the matrix equation

$$V^\pi = R^\pi + \gamma P^\pi V^\pi \implies V^\pi = (I - \gamma P^\pi)^{-1} R^\pi.$$

We compute this exactly for the uniform random policy $\pi(a\mid s) = 1/4$:


In [ ]:
S = env.NSTATES
A = env.NACTIONS

# Uniform random policy: pi[s, a] = 1/4
pi_uniform = np.ones((S, A)) / A                           # <1>

# Induced transition matrix and reward vector
P_pi = np.einsum("sa,sas->ss", pi_uniform, env.P)          # <2>
R_pi = (pi_uniform * env.R).sum(axis=1)                    # <3>

# Exact solution: V = (I - gamma * P_pi)^{-1} R_pi
I = np.eye(S)
V_exact = np.linalg.solve(I - env.gamma * P_pi, R_pi)      # <4>

print("V_exact (uniform random policy):")
print(V_exact.reshape(4, 4).round(2))


1. Each row of `pi_uniform` is a probability distribution over the 4 actions.
2. $P^\pi[s, s'] = \sum_a \pi(a\mid s)\, P(s'\mid s,a).$ The `einsum` contracts the action dimension.
3. $R^\pi[s] = \sum_a \pi(a\mid s)\, R(s,a)$ — the expected immediate reward under the policy.
4. `np.linalg.solve` solves the linear system $(I - \gamma P^\pi) V = R^\pi$ without explicitly forming the inverse, which is numerically preferable.


**Figure.** We plot $V^\pi$ for the uniform random policy as a heatmap. Green cells have higher value (better expected return); red cells have lower value:


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(4, 4))
env.render_values(V_exact, title="$V^\\pi$ — uniform random policy", ax=ax)
plt.colorbar(ax.images[0], ax=ax)
plt.tight_layout()
plt.show();


## Dynamic Programming


When the model $(P, R)$ is known, we can compute the optimal policy exactly using **dynamic programming** (DP). The central idea is **generalized policy iteration** (GPI): alternating between two operations — (1) **policy evaluation**, which computes $V^\pi$ for the current $\pi$, and (2) **policy improvement**, which constructs a new policy greedy with respect to $V^\pi.$ Iterating until convergence is guaranteed to reach $\pi^*$ for finite MDPs.


**Policy evaluation.** We implement iterative Bellman updates, sweeping over all states until $\max_s |V_{\text{new}}(s) - V(s)| < \text{tol}$:


In [ ]:
def policy_evaluation(env, pi, tol=1e-6, max_iter=1000):
    """
    Iterative policy evaluation: compute V^pi via Bellman expectation updates.
    pi: (NSTATES, NACTIONS) array of action probabilities.
    Returns V of shape (NSTATES,).
    """
    V = np.zeros(env.NSTATES)
    for i in range(max_iter):
        # Bellman expectation update (vectorized over states)
        Q = env.R + env.gamma * (env.P @ V)         # (S, A)
        V_new = (pi * Q).sum(axis=1)                # (S,)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return V_new


**Policy improvement.** Given $V^\pi$, we compute $Q^\pi(s, a)$ for all $(s, a)$ and select the greedy action:


In [ ]:
def policy_improvement(env, V):
    """
    Greedy policy improvement: return deterministic policy argmax_a Q(s,a).
    Returns pi of shape (NSTATES,) with integer action indices.
    """
    Q = env.R + env.gamma * (env.P @ V)   # (S, A)
    return Q.argmax(axis=1)               # (S,)


**Policy iteration.** We alternate evaluation and improvement until the policy stabilizes, saving snapshots every two iterations for visualization:


In [ ]:
def policy_iteration(env, tol=1e-6):
    """
    Full policy iteration loop.
    Returns (V_star, pi_star, snapshots) where snapshots is a list of
    (iteration, V, pi_deterministic) tuples.
    """
    # Start with uniform random policy
    pi = np.ones((env.NSTATES, env.NACTIONS)) / env.NACTIONS   # <1>
    snapshots = []

    for iteration in range(100):
        V = policy_evaluation(env, pi, tol=tol)
        pi_det = policy_improvement(env, V)                      # <2>

        # Snapshot: store every other iteration
        if iteration % 2 == 0 or iteration == 0:
            snapshots.append((iteration, V.copy(), pi_det.copy()))

        # Convert deterministic policy to stochastic for next evaluation
        pi_new = np.eye(env.NACTIONS)[pi_det]                    # <3>

        if np.array_equal(pi_new, pi):
            break
        pi = pi_new

    # Add final iteration to snapshots if not already there
    if snapshots[-1][0] != iteration:
        snapshots.append((iteration, V.copy(), pi_det.copy()))

    return V, pi_det, snapshots


V_pi_star, pi_pi_star, snapshots = policy_iteration(env)
print(f"Policy iteration converged. Final iterations: {snapshots[-1][0] + 1}")
print("Optimal V* (policy iteration):")
print(V_pi_star.reshape(4, 4).round(2))


1. Policy iteration can start from any policy. We use the uniform random policy.
2. `policy_improvement` returns a deterministic policy (integer action per state).
3. We convert back to a stochastic matrix by one-hot encoding the greedy actions so that `policy_evaluation` can consume it uniformly.


**Figure.** Evolution of the policy across selected iterations:


In [ ]:
#| code-fold: true
n_show = min(3, len(snapshots))
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
if n_show == 1:
    axes = [axes]
for ax, (it, V_snap, pi_snap) in zip(axes, snapshots[:n_show]):
    env.render_policy(pi_snap, V=V_snap, title=f"Iteration {it+1}", ax=ax)
plt.suptitle("Policy Iteration — Selected Iterations", fontsize=11, y=1.02)
plt.tight_layout()
plt.show();


**Value iteration.** Value iteration collapses the evaluation-improvement cycle into a single Bellman optimality update per sweep:

$$V_{k+1}(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'\mid s,a)\, V_k(s')\right].$$

This is strictly simpler to implement and converges to $V^*$ in the limit:


In [ ]:
def value_iteration(env, tol=1e-6, max_iter=1000):
    """
    Value iteration: repeated Bellman optimality updates until convergence.
    Returns (V_star, pi_star).
    """
    V = np.zeros(env.NSTATES)
    for _ in range(max_iter):
        Q = env.R + env.gamma * (env.P @ V)   # (S, A)
        V_new = Q.max(axis=1)                  # Bellman optimality update
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    pi_star = Q.argmax(axis=1)
    return V_new, pi_star


V_vi_star, pi_vi_star = value_iteration(env)
print("Optimal V* (value iteration):")
print(V_vi_star.reshape(4, 4).round(2))


**Comparison.** We verify that policy iteration and value iteration agree on both the value function and the policy:


In [ ]:
v_diff = np.max(np.abs(V_pi_star - V_vi_star))
pi_agree = np.array_equal(pi_pi_star, pi_vi_star)
print(f"Max |V_PI - V_VI|: {v_diff:.2e}")
print(f"Policies identical: {pi_agree}")


**Figure.** Optimal value function and policy from value iteration:


In [ ]:
#| code-fold: true
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
env.render_values(V_vi_star, title="Optimal $V^*$", ax=axes[0])
plt.colorbar(axes[0].images[0], ax=axes[0])
env.render_policy(pi_vi_star, V=V_vi_star, title="Optimal policy $\\pi^*$", ax=axes[1])
plt.tight_layout()
plt.show();


:::{.callout-note}
Dynamic programming requires a complete model ($P$ and $R$). In most real problems — including LLM alignment — we do not have access to the environment dynamics. Model-free methods, which learn from sampled trajectories alone, are what we turn to next.

:::


## Model-Free Prediction


In the **model-free** setting, the agent has no access to $P$ or $R$ — it can only observe transitions $(S_t, A_t, R_{t+1}, S_{t+1})$ by interacting with the environment. The **prediction** problem is to estimate $V^\pi$ for a given policy $\pi$ from such samples alone. We compare two classical methods.


### Monte Carlo Prediction


**Monte Carlo** (MC) prediction estimates $V^\pi(s)$ by averaging the actual returns $G_t$ observed from each visit to state $s.$ Using the incremental update form, after each complete episode:

$$V(S_t) \leftarrow V(S_t) + \alpha\left[G_t - V(S_t)\right].$$

MC is **unbiased** — each update is a genuine sample of $G_t$ — but can have high variance, since $G_t$ accumulates randomness over the entire episode trajectory.


### TD(0) Prediction


**Temporal difference** (TD) learning updates after every step, bootstrapping the value estimate of the next state rather than waiting for the episode to end:

$$V(S_t) \leftarrow V(S_t) + \alpha\left[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)\right].$$

The quantity $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the **TD error**, the difference between the bootstrapped target $R_{t+1} + \gamma V(S_{t+1})$ and the current estimate. TD(0) is **biased** (because $V(S_{t+1})$ is an estimate, not the truth) but typically has lower variance than MC due to the shorter effective horizon.


**MC and TD(0) implementations.** We estimate $V^\pi$ for the uniform random policy using both methods and track convergence against the exact $V^\pi$ from Section 3:


In [ ]:
def run_episode(env, policy_fn, max_steps=200):
    """Run one episode under policy_fn. Returns list of (s, a, r, s_next, done)."""
    trajectory = []
    s = env.reset(start=0)
    for _ in range(max_steps):
        a = policy_fn(s)
        s_next, r, done = env.step(s, a)
        trajectory.append((s, a, r, s_next, done))
        s = s_next
        if done:
            break
    return trajectory


def mc_prediction(env, policy_fn, n_episodes=5000, alpha=0.05, gamma=None):
    """Every-visit MC prediction. Returns V estimate and convergence errors."""
    gamma = gamma or env.gamma
    V = np.zeros(env.NSTATES)
    errors = []
    for ep in range(n_episodes):
        traj = run_episode(env, policy_fn)
        G = 0.0
        for s, a, r, s_next, done in reversed(traj):   # <1>
            G = r + gamma * G
            V[s] += alpha * (G - V[s])
        errors.append(np.max(np.abs(V - V_exact)))
    return V, errors


def td0_prediction(env, policy_fn, n_episodes=5000, alpha=0.05, gamma=None):
    """TD(0) prediction. Returns V estimate and convergence errors."""
    gamma = gamma or env.gamma
    V = np.zeros(env.NSTATES)
    errors = []
    for ep in range(n_episodes):
        traj = run_episode(env, policy_fn)
        for s, a, r, s_next, done in traj:              # <2>
            td_error = r + gamma * V[s_next] * (1 - done) - V[s]
            V[s] += alpha * td_error
        errors.append(np.max(np.abs(V - V_exact)))
    return V, errors


uniform_policy = lambda s: np.random.randint(env.NACTIONS)
np.random.seed(42)
V_mc, mc_errors = mc_prediction(env, uniform_policy, n_episodes=5000)
np.random.seed(42)
V_td, td_errors = td0_prediction(env, uniform_policy, n_episodes=5000)
print(f"MC  final max error: {mc_errors[-1]:.4f}")
print(f"TD0 final max error: {td_errors[-1]:.4f}")


1. MC requires the complete episode before computing returns. We traverse the trajectory in reverse to efficiently accumulate discounted rewards $G_t = R_{t+1} + \gamma G_{t+1}.$
2. TD(0) updates online after each step. Terminal states contribute zero future value, handled by the $(1 - \text{done})$ mask on $V(S_{t+1}).$


**Figure.** Convergence of MC and TD(0) to the exact $V^\pi$ (max absolute error over all states, plotted with a smoothed moving average):


In [ ]:
#| code-fold: true
def smooth(x, w=100):
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(smooth(mc_errors), color="C0", linewidth=1.5, label="MC (every-visit)")
ax.plot(smooth(td_errors), color="C1", linewidth=1.5, label="TD(0)")
ax.set_xlabel("Episode")
ax.set_ylabel("Max |V̂ - V_exact|")
ax.grid(linestyle="dotted", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show();


:::{.callout-note}
MC is unbiased but high-variance: each update uses a noisy full-trajectory return. TD(0) is biased — the bootstrap target depends on the current (incorrect) $V$ estimate — but lower-variance because it uses only a single step of actual reward. In practice, $\text{TD}(\lambda)$ interpolates between the two extremes via eligibility traces.

:::


## Model-Free Control — Q-Learning


The **control** problem is to learn the optimal policy from experience alone, without access to $P$ or $R.$ We focus on **Q-learning**, an off-policy TD method that directly estimates $Q^*$:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t)\right].$$

Q-learning is **off-policy** because the update uses the greedy $\max_{a'} Q(S_{t+1}, a')$ regardless of what action was actually taken from $S_{t+1}.$ This means the behavior policy (used to collect data) can differ from the target policy being optimized — a property exploited by experience replay in DQN.

For comparison, **SARSA** is an on-policy variant that uses the actual next action $A_{t+1} \sim \pi(\cdot \mid S_{t+1})$ in the update, making the learned $Q$ consistent with the behavior policy rather than the greedy one.


**Q-learning agent.** We implement an $\varepsilon$-greedy Q-learning agent:


In [ ]:
class QLearningAgent:
    """Tabular Q-learning with epsilon-greedy exploration."""

    def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=0.1):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.Q = np.zeros((env.NSTATES, env.NACTIONS))    # <1>

    def act(self, s):
        """Epsilon-greedy action selection."""
        if np.random.rand() < self.epsilon:               # <2>
            return np.random.randint(self.env.NACTIONS)
        return int(self.Q[s].argmax())

    def update(self, s, a, r, s_next, done):
        """Q-learning update."""
        max_q_next = 0.0 if done else self.Q[s_next].max()  # <3>
        target = r + self.gamma * max_q_next
        self.Q[s, a] += self.alpha * (target - self.Q[s, a])

    @property
    def greedy_policy(self):
        """Current greedy policy: argmax_a Q(s, a) for each s."""
        return self.Q.argmax(axis=1)


1. The $Q$-table is initialized to zero — a common choice that encourages early exploration via optimistic initialization relative to the true (negative) values.
2. With probability $\varepsilon$ the agent explores by selecting a uniformly random action; otherwise it exploits the current greedy action.
3. At terminal states, the bootstrapped target reduces to the immediate reward $r$ alone — there is no next-state value to add.


**Training.** We train for 5,000 episodes and record the episodic return:


In [ ]:
np.random.seed(42)
agent = QLearningAgent(env, alpha=0.1, gamma=0.99, epsilon=0.15)

N_EPISODES = 5000
MAX_STEPS = 200
episode_returns = []

for ep in range(N_EPISODES):
    s = env.reset(start=0)
    total_r = 0.0
    for _ in range(MAX_STEPS):
        a = agent.act(s)
        s_next, r, done = env.step(s, a)
        agent.update(s, a, r, s_next, done)
        total_r += r
        s = s_next
        if done:
            break
    episode_returns.append(total_r)

print(f"Mean return (last 500 eps): {np.mean(episode_returns[-500:]):.2f}")


**Figure.** Learning curve (episode return, smoothed over 200 episodes):


In [ ]:
#| code-fold: true
def smooth(x, w=200):
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(smooth(episode_returns), color="C0", linewidth=1.5, label="Return (smoothed)")
ax.axhline(np.mean(episode_returns[-500:]), color="gray", linestyle="dashed",
           lw=1.0, label=f"Final mean = {np.mean(episode_returns[-500:]):.1f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.grid(linestyle="dotted", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show();


**Verification.** We check that the Q-learned greedy policy matches the DP optimal policy:


In [ ]:
pi_ql = agent.greedy_policy
V_ql = agent.Q.max(axis=1)   # V(s) = max_a Q(s, a)

match = np.mean(pi_ql == pi_vi_star)
v_err = np.max(np.abs(V_ql - V_vi_star))
print(f"Policy agreement with DP: {match:.0%} of states")
print(f"Max |V_QL - V_DP|:        {v_err:.4f}")


**Figure.** Q-learned optimal policy overlaid on the learned $V^*$:


In [ ]:
#| code-fold: true
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
env.render_values(V_ql, title="$V^*$ — Q-learning", ax=axes[0])
plt.colorbar(axes[0].images[0], ax=axes[0])
env.render_policy(pi_ql, V=V_ql, title="Policy — Q-learning", ax=axes[1])
plt.tight_layout()
plt.show();


## Function Approximation — Why Tables Don't Scale


Tabular methods maintain a separate value estimate for every state (or state-action pair). This is perfectly tractable for a $4 \times 4$ GridWorld with 16 states, but collapses immediately in realistic settings. A $100 \times 100$ grid has $10^4$ states. The Atari game space has approximately $10^{60}$ distinct frames. A language model operating on a context of 2,048 tokens drawn from a vocabulary of 50,000 has a state space of $50{,}000^{2048}$ — a number with more digits than there are atoms in the observable universe.

The practical solution is **function approximation**: parameterize $V(s; \theta)$ or $Q(s, a; \theta)$ as a neural network and update $\theta$ via gradient descent. DQN [@mnih2015human] showed this could work for Atari by adding experience replay and a target network to stabilize training. However, deep Q-networks are not on the most direct path to modern LLM alignment — that path goes through **policy gradient** methods, which directly optimize $\mathbb{E}_\pi[G_0]$ by differentiating through sampled trajectories. A forthcoming notebook builds neural network policies and derives the policy gradient theorem. The combination — a parameterized critic $V(s; \theta_v)$ and a parameterized actor $\pi(a \mid s; \theta_\pi)$ — is the **actor-critic** architecture underlying every modern alignment algorithm.

**Remark.** One technical hazard deserves mention. The combination of (1) function approximation, (2) bootstrapping (TD-style updates using $V(s')$ as a target), and (3) off-policy data is sometimes called the **deadly triad** [@sutton2018reinforcement]. Each ingredient alone is harmless; together they can cause the value estimates to diverge. DQN sidestepped the off-policy problem with careful design choices. Policy gradient methods largely avoid it by staying on-policy.


## Appendix: RL Algorithm Taxonomy


Reinforcement learning algorithms can be organized along three independent dimensions. The table below places the methods from this notebook and the rest of the series within this taxonomy:

<br>

| Dimension | Options | Examples |
|-----------|---------|----------|
| **Model** | Model-based | Dyna, AlphaZero, World Models |
|           | Model-free | Q-learning, TD, MC, policy gradients |
| **Policy** | Value-based | Q-learning (tabular), DQN |
|            | Policy-based | REINFORCE, PPO, GRPO |
|            | Actor-Critic | A3C, SAC, PPO (with critic), RLHF |
| **Data** | On-policy | SARSA, Monte Carlo, PPO, GRPO |
|          | Off-policy | Q-learning, DQN, SAC, experience replay |

: {tbl-colwidths="[20, 30, 50]"}

<br>

This notebook covers model-free, value-based, and both on-policy (MC, TD) and off-policy (Q-learning) methods. The next notebook in the series builds policy gradient methods: model-free, policy-based, and on-policy — the direct ancestors of GRPO.


---


■
